In [1]:
try:
    import ydf
except:
    !uv pip install ydf
    import ydf

print(f"ydf version: {ydf.__version__}")

ydf version: 0.15.0


In [2]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", 500)

In [3]:
# !uv pip install -U scikit-learn

In [4]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier

# 1. LOAD DATA

In [5]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e9/train.csv'
TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e9/test.csv'
SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv'
ORIG_PATH  = '/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/EV_Adoption_and_Range_Anxiety_Dataset.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
orig  = pd.read_csv(ORIG_PATH)
submission = pd.read_csv(SUB_PATH)

# 2.  FEATURE ENGINEERING

In [6]:
TARGET = 'Will_Buy_EV'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
orig[TARGET] = orig[TARGET].map({'Yes': 1, 'No': 0})

train['is_train'] = 1
test['is_train'] = 0
test[TARGET] = np.nan
combined = pd.concat([train, test], ignore_index=True)
combined.drop(columns=['Number_of_Cars_Owned'], inplace=True, errors='ignore')

cat_cols = combined.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = [c for c in combined.columns if c not in cat_cols + ['id', 'is_train', TARGET]]

# digits_cols = []
# for col in ["Annual_Income_USD", "Daily_Commute_km"]:
#     scaled = np.rint(combined[col] * 10**4).astype("int64")
#     for k in range(-4, 4):
#         dig_col = f"{col}_digit{k}"
#         combined[dig_col] = (scaled // 10**(k + 4) % 10).astype('int8')
#         digits_cols.append(dig_col)
        
# Extract digits from the 10^-4 place up to the 10^3 place
digit_features = []
for c in num_cols:
    scaled = np.rint(combined[c] * 10**4).astype("int64")
    for k in range(-4, 4):
        col_name = f"{c}_digit{k}"
        combined[col_name] = (scaled // 10**(k + 4) % 10).astype('int8')
        # combined[col_name] = (combined[c].fillna(0) // (10**k) % 10).astype('int8')
        digit_features.append(col_name)

# Add the new digit features so they get processed by your frequency/target encoders
num_cols.extend(digit_features)

# Map Original Dataset Target Means
orig_global_mean = orig[TARGET].mean()
for col in cat_cols + num_cols:
    if col in orig.columns:
        real_world_stats = orig.groupby(col, observed=False)[TARGET].mean()
        combined[f"{col}_org_mean"] = combined[col].map(real_world_stats).fillna(orig_global_mean).astype(float)

# Convert Numerics to String Categories
num_to_cat_cols = []
for col in num_cols:
    cat_name = f"{col}_cat"
    combined[cat_name] = combined[col].fillna('NaN').astype(str)
    num_to_cat_cols.append(cat_name)

# Global Frequency Encoding
all_cats = cat_cols + num_to_cat_cols
for col in all_cats:
    freq_mapping = combined[col].value_counts(normalize=True).to_dict()
    combined[f"{col}_fe"] = combined[col].map(freq_mapping).astype(float).fillna(0.0)

# The Mode Collapse Spike
# combined['is_30k_spike'] = (combined['Annual_Income_USD'] == 30000.0).astype('int8')  # Pruned! the XGBoost model do need it
# The Millionaire Cliff (100% buy rate region)
# combined['is_millionaire_cliff'] = (combined['Annual_Income_USD'] >= 170537.0).astype('int8')  # Pruned! the XGBoost model do need it
# The Dead Zone (0% buy rate region)
# combined['is_dead_zone'] = ((combined['Annual_Income_USD'] >= 38000.0) & (combined['Annual_Income_USD'] <= 42000.0)).astype('int8')  # Pruned! the XGBoost model do need it
# Environmental Concern Extremes
combined['is_env_hater'] = (combined['Environmental_Concern_Level'] == 1).astype('int8')

# Markus's "Smooth Keys" (Binned Numerics)
combined['income_exact_int'] = np.floor(combined['Annual_Income_USD']).astype(str)
combined['income100_floor']  = np.floor(combined['Annual_Income_USD'] / 100.0).astype(str)
combined['income1000_floor'] = np.floor(combined['Annual_Income_USD'] / 1000.0).astype(str)
combined['commute_integer']  = np.floor(combined['Daily_Commute_km']).astype(str)
# Adding these 4 new string columns to all_cats so they get Frequency and Target Encoded
all_cats.extend(['income_exact_int', 'income100_floor', 'income1000_floor', 'commute_integer'])

train = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test = combined[combined['is_train'] == 0].drop(columns=['is_train', TARGET])

#  FEATURE DROPPING
# Identify numeric columns to evaluate for correlatio (ignore strings/objects because .corr() will fail on them)
eval_cols = [c for c in train.columns if c not in ['id', TARGET] and pd.api.types.is_numeric_dtype(train[c])]

# Find perfectly correlated features (1.0 correlation)
corr_matrix = train[eval_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [column for column in upper_tri.columns if any(upper_tri[column] == 1.0)]

# Find constant features (only 1 unique value) in train or test
to_drop_const = [c for c in train.columns if train[c].nunique() == 1] + \
                [c for c in test.columns if test[c].nunique() == 1]

# Combine all bad features into a set to drop
DROP = set(to_drop_corr).union(set(to_drop_const))
DROP = [c for c in DROP if c not in ['id', TARGET]] 

if len(DROP) > 0:
    print(f"   -> Dropping {len(DROP)} redundant/constant features: {DROP}")
    train.drop(columns=DROP, inplace=True, errors='ignore')
    test.drop(columns=DROP, inplace=True, errors='ignore')

FEATURES = [c for c in test.columns if c != 'id']
TARGET_ENCODE_COLS = [c for c in all_cats if c not in DROP] 

print(f"✅ Total Features: {len(FEATURES)}")
print(f"✅ Columns to Target Encode: {len(TARGET_ENCODE_COLS)}")

train

   -> Dropping 104 redundant/constant features: ['Daily_Commute_km_digit-3_cat_fe', 'Age_digit-2_cat_fe', 'Charging_Stations_Near_Home_digit-2_cat', 'Environmental_Concern_Level_digit1', 'Age_digit-4', 'Charging_Stations_Near_Work_digit3', 'Charging_Stations_Near_Work_digit2_cat_fe', 'Daily_Commute_km_digit2_cat_fe', 'Charging_Stations_Near_Work_digit-4_cat_fe', 'Daily_Commute_km_digit2', 'Charging_Stations_Near_Work_digit-2_cat', 'Environmental_Concern_Level_digit-1_cat', 'Charging_Stations_Near_Home_digit3_cat', 'Annual_Income_USD_digit-2_cat_fe', 'Age_digit3_cat', 'Charging_Stations_Near_Home_digit2_cat', 'Charging_Stations_Near_Home_digit-4', 'Daily_Commute_km_digit-3_cat', 'Environmental_Concern_Level_digit-4_cat_fe', 'Charging_Stations_Near_Home_digit-1', 'Environmental_Concern_Level_digit-1', 'Annual_Income_USD_digit-1', 'Age_digit-4_cat', 'Charging_Stations_Near_Work_digit-1_cat', 'Daily_Commute_km_digit-4_cat', 'Charging_Stations_Near_Home_digit-2', 'Environmental_Concern_Leve

,id,Age,Annual_Income_USD,Daily_Commute_km,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV,Age_digit0,Age_digit1,Annual_Income_USD_digit0,Annual_Income_USD_digit1,Annual_Income_USD_digit2,Annual_Income_USD_digit3,Daily_Commute_km_digit-1,Daily_Commute_km_digit0,Daily_Commute_km_digit1,Charging_Stations_Near_Home_digit0,Charging_Stations_Near_Home_digit1,Charging_Stations_Near_Work_digit0,Charging_Stations_Near_Work_digit1,Gender_org_mean,City_Type_org_mean,Current_Car_Type_org_mean,Home_Charging_Possible_org_mean,Subsidy_Available_org_mean,Range_Anxiety_Level_org_mean,Age_org_mean,Annual_Income_USD_org_mean,Daily_Commute_km_org_mean,Charging_Stations_Near_Home_org_mean,Charging_Stations_Near_Work_org_mean,Environmental_Concern_Level_org_mean,Age_cat,Annual_Income_USD_cat,Daily_Commute_km_cat,Charging_Stations_Near_Home_cat,Charging_Stations_Near_Work_cat,Environmental_Concern_Level_cat,Age_digit0_cat,Age_digit1_cat,Annual_Income_USD_digit0_cat,Annual_Income_USD_digit1_cat,Annual_Income_USD_digit2_cat,Annual_Income_USD_digit3_cat,Daily_Commute_km_digit-1_cat,Daily_Commute_km_digit0_cat,Daily_Commute_km_digit1_cat,Charging_Stations_Near_Home_digit0_cat,Charging_Stations_Near_Home_digit1_cat,Charging_Stations_Near_Work_digit0_cat,Charging_Stations_Near_Work_digit1_cat,Environmental_Concern_Level_digit0_cat,Gender_fe,City_Type_fe,Current_Car_Type_fe,Home_Charging_Possible_fe,Subsidy_Available_fe,Range_Anxiety_Level_fe,Age_cat_fe,Annual_Income_USD_cat_fe,Daily_Commute_km_cat_fe,Charging_Stations_Near_Home_cat_fe,Charging_Stations_Near_Work_cat_fe,Environmental_Concern_Level_cat_fe,Age_digit0_cat_fe,Age_digit1_cat_fe,Annual_Income_USD_digit0_cat_fe,Annual_Income_USD_digit1_cat_fe,Annual_Income_USD_digit2_cat_fe,Annual_Income_USD_digit3_cat_fe,Daily_Commute_km_digit-1_cat_fe,Daily_Commute_km_digit0_cat_fe,Daily_Commute_km_digit1_cat_fe,Charging_Stations_Near_Home_digit0_cat_fe,Charging_Stations_Near_Home_digit1_cat_fe,Charging_Stations_Near_Work_digit0_cat_fe,Charging_Stations_Near_Work_digit1_cat_fe,is_env_hater,income_exact_int,income100_floor,income1000_floor,commute_integer
0,0,66,92887.0,23.4,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,0.0,6,6,7,8,8,2,4,3,2,3,0,7,0,0.170929,0.176817,0.172250,0.200714,0.025301,0.198676,0.200913,0.000000,0.181818,0.154994,0.166913,0.021361,66,92887.0,23.4,3,7,1.0,6,6,7,8,8,2,4,3,2,3,0,7,0,1,0.550232,0.382160,0.453965,0.69214,0.371583,0.903671,0.022189,0.000156,0.001025,0.088105,0.067977,0.220470,0.119046,0.218299,0.089344,0.098165,0.136997,0.114493,0.089187,0.068507,0.155343,0.119869,0.839831,0.095176,0.742066,1,92887.0,928.0,92.0,23.0
1,1,38,30000.0,5.0,2,2,4.0,Male,Rural,SUV,Yes,No,Low,0.0,8,3,0,0,0,0,0,5,0,2,0,2,0,0.170929,0.189840,0.181010,0.200714,0.025301,0.198676,0.175355,0.066427,0.191517,0.187404,0.188034,0.252891,38,30000.0,5.0,2,2,4.0,8,3,0,0,0,0,0,5,0,2,0,2,0,4,0.550232,0.185583,0.368738,0.69214,0.371583,0.903671,0.020044,0.091999,0.215599,0.151632,0.074310,0.195549,0.107725,0.218350,0.199227,0.149168,0.146475,0.158213,0.292327,0.290030,0.218171,0.183997,0.839831,0.100067,0.742066,0,30000.0,300.0,30.0,5.0
2,2,26,94389.0,36.8,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,1.0,6,2,9,8,3,4,8,6,3,8,0,5,1,0.180302,0.169197,0.172250,0.133698,0.274467,0.198676,0.200000,1.000000,0.285714,0.195710,0.166667,0.414046,26,94389.0,36.8,8,15,5.0,6,2,9,8,3,4,8,6,3,8,0,5,1,5,0.441876,0.432257,0.453965,0.30786,0.628417,0.903671,0.022353,0.000063,0.002892,0.031121,0.024436,0.191398,0.119046,0.108970,0.090048,0.098165,0.073144,0.109598,0.076601,0.083534,0.184213,0.031121,0.839831,0.094157,0.257934,0,94389.0,943.0,94.0,36.0
3,3,66,73580.0,23.7,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,0.0,6,6,0,8,5,3,7,3,2,6,0,9,0,0.170929,0.176817,0.174655,0.200714,0.025301,0.198676,0.200913,1.000000,0.300000,0.159174,0.178470,0.144535,66,73580.0,23.7,6,9,3.0,6,6,0,8,5,3,7,3,2,6,0,9,0,3,0.5502

In [7]:
# ==========================================
# MANUAL FEATURE PRUNING (Calculated via Gain Thresholding)
# ==========================================
print("✂️ Feature for dynamic pruning (Dropping 88 low-gain features)...")

features_to_drop = [
    'Age', 'Daily_Commute_km_digit-3_cat_TE_100', 'Charging_Stations_Near_Home_digit0',
    'Daily_Commute_km_digit1_cat_TE_100', 'Annual_Income_USD_digit1', 'Charging_Stations_Near_Work_digit1_cat_TE_auto',
    'Daily_Commute_km_digit1_cat_TE_auto', 'Daily_Commute_km_digit0_cat_TE_10', 'Gender_org_mean',
    'Daily_Commute_km_digit-4_cat_TE_100', 'Daily_Commute_km_digit0_cat_TE_auto', 'Daily_Commute_km_digit-4_cat_TE_auto',
    'Daily_Commute_km_cat_fe', 'Annual_Income_USD_digit1_cat_fe', 'Charging_Stations_Near_Work_digit0_cat_TE_100',
    'Annual_Income_USD_digit2_cat_TE_100', 'Annual_Income_USD_digit0', 'Charging_Stations_Near_Home_digit-3_cat_TE_auto',
    'Gender_TE_100', 'Current_Car_Type_TE_10', 'Age_digit1_cat_fe', 'Charging_Stations_Near_Work_digit1_cat_TE_100',
    'Annual_Income_USD_digit0_cat_TE_10', 'Daily_Commute_km_digit-3_cat_TE_10', 'Charging_Stations_Near_Work_digit0_cat_TE_10',
    'Gender_TE_auto', 'Daily_Commute_km_digit0_cat_TE_100', 'Charging_Stations_Near_Home_digit-4_cat_TE_auto',
    'Daily_Commute_km_digit-1_cat_TE_10', 'Charging_Stations_Near_Work_digit0_cat_TE_auto', 'Daily_Commute_km_digit-1',
    'Charging_Stations_Near_Work_digit0', 'Annual_Income_USD_digit0_cat_TE_auto', 'Daily_Commute_km_digit-4_cat_TE_10',
    'Charging_Stations_Near_Work_digit-3_cat_TE_auto', 'Charging_Stations_Near_Work_org_mean', 'Age_digit0_cat_TE_auto',
    'Age_digit0_cat_fe', 'Annual_Income_USD_digit0_cat_fe', 'Charging_Stations_Near_Home_digit-4_cat_TE_100',
    'Daily_Commute_km_digit-3_cat_TE_auto', 'Age_digit0', 'Annual_Income_USD_digit2_cat_TE_auto',
    'Annual_Income_USD_digit0_cat_TE_100', 'Daily_Commute_km_org_mean', 'Age_digit0_cat_TE_10',
    'Current_Car_Type_TE_auto', 'Current_Car_Type_TE_100', 'Age_digit1_cat_TE_auto', 'Charging_Stations_Near_Home_digit-4',
    'Annual_Income_USD_digit2_cat_TE_10', 'Daily_Commute_km_digit-1_cat_TE_100', 'Age_cat_fe',
    'Daily_Commute_km_digit0_cat_fe', 'Daily_Commute_km_digit-1_cat_fe', 'Daily_Commute_km_digit-1_cat_TE_auto',
    'Daily_Commute_km_digit-2_cat_TE_10', 'Age_digit1', 'Age_digit1_cat_TE_100', 'Gender_TE_10', 'Age_digit1_cat_TE_10',
    'Age_digit0_cat_TE_100', 'Daily_Commute_km_digit-3', 'Charging_Stations_Near_Home_digit-3_cat_TE_100', 'Gender_fe',
    'Charging_Stations_Near_Home_digit-3_cat_TE_10', 'Daily_Commute_km_digit-4_cat_fe', 'Daily_Commute_km_digit-4',
    'Daily_Commute_km_digit-3_cat_fe', 'Charging_Stations_Near_Work_digit-3_cat_TE_10', 'Daily_Commute_km_digit-2_cat_TE_100',
    'Charging_Stations_Near_Home_digit-2_cat_TE_auto', 'Charging_Stations_Near_Home_digit-1_cat_TE_auto',
    'Charging_Stations_Near_Work_digit1_cat_fe', 'Charging_Stations_Near_Work_digit-2_cat_TE_100',
    'Charging_Stations_Near_Home_digit-2_cat_TE_100', 'Charging_Stations_Near_Work_digit-3_cat_TE_100', 'is_dead_zone',
    'is_millionaire_cliff', 'is_30k_spike', 'Charging_Stations_Near_Home_digit-1_cat_TE_100',
    'Charging_Stations_Near_Home_digit-1_cat_TE_10', 'Charging_Stations_Near_Home_digit-2_cat_TE_10',
    'Charging_Stations_Near_Work_digit-1_cat_TE_10', 'Charging_Stations_Near_Work_digit-2_cat_TE_auto',
    'Charging_Stations_Near_Work_digit-1_cat_TE_auto', 'Charging_Stations_Near_Work_digit-1_cat_TE_100',
    'Charging_Stations_Near_Work_digit-2_cat_TE_10'
]

✂️ Feature for dynamic pruning (Dropping 88 low-gain features)...


In [8]:
# %%time

# X = train[FEATURES]
# train[TARGET] = train[TARGET].astype(int)

# ## -- Search HPO space for 'LOCAL' & 'BEST_FIRST_GLOBAL' --

# tuner = ydf.RandomSearchTuner(num_trials=20, automatic_search_space=False)
# # tuner.choice('num_trees', [5000, 2000])
# # tuner.choice('shrinkage', [0.02, 0.05]) 
# # tuner.choice('subsample', [0.9, 1.0])
# tuner.choice('categorical_algorithm', ['CART', 'RANDOM']) #'CART', 'RANDOM', 'ONE_HOT'
# tuner.choice('l2_regularization', [0.0, 0.1, 0.5, 1.0])

# ## -- 'LOCAL' acts like 'depthwise': uses 'max_depth' --
# local_subspace = tuner.choice('growing_strategy', ['LOCAL'])
# local_subspace.choice('max_depth', [4, 6])

# ## -- 'BEST_FIRST_GLOBAL' acts like 'lossguide': uses 'max_num_nodes' --
# global_subspace = tuner.choice('growing_strategy', ['BEST_FIRST_GLOBAL'], merge=True)
# # global_subspace.choice('max_num_nodes', [None, 32, 64, 128])

# # ## -- Enable ydf process numeric features as categories --
# # num_as_cat = []
# # for f in CATS:
# #     num_as_cat.append(ydf.Feature(f, ydf.Semantic.CATEGORICAL))

# ydf_HPO_model = ydf.GradientBoostedTreesLearner(
#     task=ydf.Task.CLASSIFICATION,
#     label=TARGET, 
#     tuner=tuner,
#     num_trees=10_000,
#     # shrinkage=0.1,
#     # class_weights=get_class_weights(train_data[TARGET]), ## 
#     # max_depth=4,
#     # features=num_as_cat,
#     # include_all_columns=True,
#     # l2_regularization=0.15,
#     # categorical_algorithm='RANDOM',
#     # early_stopping_initial_iteration=10,
#     # early_stopping_num_trees_look_ahead=100,
#     random_seed=42, 
#     num_threads=3, 
# ).train(train[FEATURES+[TARGET]], verbose=1)

# ydf_HPO_model.describe()

## HPO SEARCH

In [9]:
%%time

## -- Search HPO space in 'LOCAL' & 'BEST_FIRST_GLOBAL' --

MODEL_NAME = 'YDFsearch'
ydf_models = []

FOLDS = 3
print(f"\n🚀 Training {MODEL_NAME} with {FOLDS} Folds...")

X = train[FEATURES]
y = train[TARGET].astype(int)
X_test = test[FEATURES]

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))

# feature_importances = pd.DataFrame()

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx].copy(), y.iloc[valid_idx]
    # X_test_fold = X_test.copy()

    # Triple Sklearn Target Encoders (Auto, Strict 10, and Massive 100)
    te_auto = TargetEncoder(shuffle=True, cv=5, smooth='auto', random_state=42)
    te_10   = TargetEncoder(shuffle=True, cv=5, smooth=10.0, random_state=42)
    te_100  = TargetEncoder(shuffle=True, cv=5, smooth=100.0, random_state=42)

    X_train_enc_auto = te_auto.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
    X_valid_enc_auto = te_auto.transform(X_valid[TARGET_ENCODE_COLS])
    # X_test_enc_auto  = te_auto.transform(X_test_fold[TARGET_ENCODE_COLS])

    X_train_enc_10 = te_10.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
    X_valid_enc_10 = te_10.transform(X_valid[TARGET_ENCODE_COLS])
    # X_test_enc_10  = te_10.transform(X_test_fold[TARGET_ENCODE_COLS])

    X_train_enc_100 = te_100.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
    X_valid_enc_100 = te_100.transform(X_valid[TARGET_ENCODE_COLS])
    # X_test_enc_100  = te_100.transform(X_test_fold[TARGET_ENCODE_COLS])

    for i, col in enumerate(TARGET_ENCODE_COLS):
        # Auto smoothing TE
        X_train[f"{col}_TE_auto"] = X_train_enc_auto[:, i].astype('float32')
        X_valid[f"{col}_TE_auto"] = X_valid_enc_auto[:, i].astype('float32')
        # X_test_fold[f"{col}_TE_auto"] = X_test_enc_auto[:, i].astype('float32')
        
        # Strict (10.0) smoothing TE
        X_train[f"{col}_TE_10"] = X_train_enc_10[:, i].astype('float32')
        X_valid[f"{col}_TE_10"] = X_valid_enc_10[:, i].astype('float32')
        # X_test_fold[f"{col}_TE_10"] = X_test_enc_10[:, i].astype('float32')

        # Massive (100.0) smoothing TE (Markus's method)
        X_train[f"{col}_TE_100"] = X_train_enc_100[:, i].astype('float32')
        X_valid[f"{col}_TE_100"] = X_valid_enc_100[:, i].astype('float32')
        # X_test_fold[f"{col}_TE_100"] = X_test_enc_100[:, i].astype('float32')
        
        # Drop the original string column
        X_train.drop(columns=[col], inplace=True)
        X_valid.drop(columns=[col], inplace=True)
        # X_test_fold.drop(columns=[col], inplace=True)
        
    # APPLY DYNAMIC PRUNING
    if len(features_to_drop) > 0:
        # Check X_train.columns because it contains both base features and the new _TE_ features
        safe_to_drop = [c for c in features_to_drop if c in X_train.columns]
        
        if len(safe_to_drop) > 0:
            if fold == 1:
                print(f"   -> Successfully pruned {len(safe_to_drop)} features.")
                
            X_train.drop(columns=safe_to_drop, inplace=True, errors='ignore')
            X_valid.drop(columns=safe_to_drop, inplace=True, errors='ignore')
            # X_test_fold.drop(columns=safe_to_drop, inplace=True, errors='ignore')

    if fold == 1:
        display(X_train.head())
    
    tuner = ydf.RandomSearchTuner(num_trials=20, automatic_search_space=False)
    # tuner.choice('num_trees', [5000, 2000])
    # tuner.choice('shrinkage', [0.02, 0.05]) 
    # tuner.choice('subsample', [0.9, 1.0])
    tuner.choice('categorical_algorithm', ['CART', 'RANDOM']) #'CART', 'RANDOM', 'ONE_HOT'
    tuner.choice('l2_regularization', [0.0, 0.1, 0.5])
    
    ## -- 'LOCAL' acts like 'depthwise': uses 'max_depth' --
    local_subspace = tuner.choice('growing_strategy', ['LOCAL'])
    local_subspace.choice('max_depth', [4, 5, 6])
    
    ## -- 'BEST_FIRST_GLOBAL' acts like 'lossguide': uses 'max_num_nodes' --
    global_subspace = tuner.choice('growing_strategy', ['BEST_FIRST_GLOBAL'], merge=True)
    # global_subspace.choice('max_num_nodes', [None, 32, 64, 128])
    
    # ## -- Enable ydf process numeric features as categories --
    # num_as_cat = []
    # for f in CATS:
    #     num_as_cat.append(ydf.Feature(f, ydf.Semantic.CATEGORICAL))
    
    ydf_HPO_model = ydf.GradientBoostedTreesLearner(
        task=ydf.Task.CLASSIFICATION,
        label=TARGET, 
        tuner=tuner,
        num_trees=1000,
        shrinkage=0.1,
        # class_weights=get_class_weights(train_data[TARGET]), ## 
        # max_depth=4,
        # features=num_as_cat,
        # include_all_columns=True,
        # l2_regularization=0.15,
        # categorical_algorithm='RANDOM',
        # early_stopping_initial_iteration=10,
        early_stopping_num_trees_look_ahead=50,
        random_seed=42, 
        num_threads=3, 
    ).train(pd.concat([X_train, y_train], axis=1), verbose=1)
    
    ydf_models.append(ydf_HPO_model)
    display(ydf_HPO_model.describe())


🚀 Training YDFsearch with 3 Folds...
   -> Successfully pruned 52 features.


,Annual_Income_USD,Daily_Commute_km,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Annual_Income_USD_digit2,Annual_Income_USD_digit3,Daily_Commute_km_digit0,Daily_Commute_km_digit1,Charging_Stations_Near_Home_digit1,Charging_Stations_Near_Work_digit1,City_Type_org_mean,Current_Car_Type_org_mean,Home_Charging_Possible_org_mean,Subsidy_Available_org_mean,Range_Anxiety_Level_org_mean,Age_org_mean,Annual_Income_USD_org_mean,Charging_Stations_Near_Home_org_mean,Environmental_Concern_Level_org_mean,City_Type_fe,Current_Car_Type_fe,Home_Charging_Possible_fe,Subsidy_Available_fe,Range_Anxiety_Level_fe,Annual_Income_USD_cat_fe,Charging_Stations_Near_Home_cat_fe,Charging_Stations_Near_Work_cat_fe,Environmental_Concern_Level_cat_fe,Annual_Income_USD_digit2_cat_fe,Annual_Income_USD_digit3_cat_fe,Daily_Commute_km_digit1_cat_fe,Charging_Stations_Near_Home_digit0_cat_fe,Charging_Stations_Near_Home_digit1_cat_fe,Charging_Stations_Near_Work_digit0_cat_fe,is_env_hater,City_Type_TE_auto,City_Type_TE_10,City_Type_TE_100,Home_Charging_Possible_TE_auto,Home_Charging_Possible_TE_10,Home_Charging_Possible_TE_100,Subsidy_Available_TE_auto,Subsidy_Available_TE_10,Subsidy_Available_TE_100,Range_Anxiety_Level_TE_auto,Range_Anxiety_Level_TE_10,Range_Anxiety_Level_TE_100,Age_cat_TE_auto,Age_cat_TE_10,Age_cat_TE_100,Annual_Income_USD_cat_TE_auto,Annual_Income_USD_cat_TE_10,Annual_Income_USD_cat_TE_100,Daily_Commute_km_cat_TE_auto,Daily_Commute_km_cat_TE_10,Daily_Commute_km_cat_TE_100,Charging_Stations_Near_Home_cat_TE_auto,Charging_Stations_Near_Home_cat_TE_10,Charging_Stations_Near_Home_cat_TE_100,Charging_Stations_Near_Work_cat_TE_auto,Charging_Stations_Near_Work_cat_TE_10,Charging_Stations_Near_Work_cat_TE_100,Environmental_Concern_Level_cat_TE_auto,Environmental_Concern_Level_cat_TE_10,Environmental_Concern_Level_cat_TE_100,Annual_Income_USD_digit1_cat_TE_auto,Annual_Income_USD_digit1_cat_TE_10,Annual_Income_USD_digit1_cat_TE_100,Annual_Income_USD_digit3_cat_TE_auto,Annual_Income_USD_digit3_cat_TE_10,Annual_Income_USD_digit3_cat_TE_100,Daily_Commute_km_digit1_cat_TE_10,Charging_Stations_Near_Home_digit0_cat_TE_auto,Charging_Stations_Near_Home_digit0_cat_TE_10,Charging_Stations_Near_Home_digit0_cat_TE_100,Charging_Stations_Near_Home_digit1_cat_TE_auto,Charging_Stations_Near_Home_digit1_cat_TE_10,Charging_Stations_Near_Home_digit1_cat_TE_100,Charging_Stations_Near_Work_digit1_cat_TE_10,Environmental_Concern_Level_digit0_cat_TE_auto,Environmental_Concern_Level_digit0_cat_TE_10,Environmental_Concern_Level_digit0_cat_TE_100,income_exact_int_TE_auto,income_exact_int_TE_10,income_exact_int_TE_100,income100_floor_TE_auto,income100_floor_TE_10,income100_floor_TE_100,income1000_floor_TE_auto,income1000_floor_TE_10,income1000_floor_TE_100,commute_integer_TE_auto,commute_integer_TE_10,commute_integer_TE_100
0,92887.0,23.4,3,7,1.0,8,2,3,2,0,0,0.176817,0.172250,0.200714,0.025301,0.198676,0.200913,0.0,0.154994,0.021361,0.382160,0.453965,0.69214,0.371583,0.903671,0.000156,0.088105,0.067977,0.220470,0.136997,0.114493,0.155343,0.119869,0.839831,0.095176,1,0.182348,0.182348,0.182343,0.195929,0.195928,0.195920,0.005810,0.005823,0.005937,0.189005,0.189004,0.189000,0.169636,0.169641,0.169697,0.258433,0.245472,0.201812,0.164164,0.164444,0.166584,0.163503,0.163506,0.163538,0.170042,0.170044,0.170061,0.005316,0.005338,0.005531,0.201178,0.201171,0.201103,0.186257,0.186254,0.186229,0.177719,0.165673,0.165675,0.165694,0.175073,0.175073,0.175073,0.176888,0.005316,0.005338,0.005531,0.258433,0.245472,0.201812,0.180894,0.180859,0.180526,0.231498,0.231427,0.230704,0.184916,0.184894,0.184677
3,73580.0,23.7,6,9,3.0,5,3,3,2,0,0,0.176817,0.174655,0.200714,0.025301,0.198676,0.200913,1.0,0.159174,0.144535,0.382160,0.118863,0.69214,0.371583,0.903671,0.000263,0.083082,0.074363,0.190380,0.118031,0.108093,0.155343,0.083082,0.839831,0.099631,0,0.181848,0.181847,0.181842,0.195639,0.195638,0.195630,0.005546,0.005559,0.005673,0.189073,0.189073,0.189069,0.168452,0

Train model on 445776 examples
Model trained in 2:20:39.671015


trial,score,duration,categorical_algorithm,l2_regularization,growing_strategy,max_depth
1,-0.439309,899.175,CART,0,LOCAL,4
13,-0.439309,5991.1,RANDOM,0,LOCAL,4
16,-0.439498,7339.97,RANDOM,0,LOCAL,5
4,-0.439498,2194.13,CART,0,LOCAL,5
15,-0.439663,6844,CART,0.1,LOCAL,5
12,-0.439663,5516.88,RANDOM,0.1,LOCAL,5
6,-0.439689,3003.5,RANDOM,0.1,LOCAL,4
19,-0.439741,8439.64,CART,0.5,LOCAL,4
10,-0.439741,4598.18,RANDOM,0.5,LOCAL,4
7,-0.439809,3400.73,CART,0.5,LOCAL,6


Train model on 445777 examples
Model trained in 2:04:38.106909


trial,score,duration,categorical_algorithm,l2_regularization,growing_strategy,max_depth
15,-0.441671,6147.05,CART,0.1,LOCAL,5
12,-0.441671,4865.52,RANDOM,0.1,LOCAL,5
3,-0.441831,1268.46,RANDOM,0.5,LOCAL,6
7,-0.441831,2765.21,CART,0.5,LOCAL,6
10,-0.442025,3991.83,RANDOM,0.5,LOCAL,4
19,-0.442025,7478.09,CART,0.5,LOCAL,4
9,-0.442039,3626.85,CART,0.1,LOCAL,6
11,-0.442039,4394.51,RANDOM,0.1,LOCAL,6
4,-0.442043,1586.88,CART,0,LOCAL,5
16,-0.442043,6467.43,RANDOM,0,LOCAL,5


Train model on 445777 examples
Model trained in 2:04:50.755649


trial,score,duration,categorical_algorithm,l2_regularization,growing_strategy,max_depth
16,-0.442205,6376.36,RANDOM,0,LOCAL,5
4,-0.442205,1787.99,CART,0,LOCAL,5
11,-0.442285,4613.66,RANDOM,0.1,LOCAL,6
9,-0.442285,3732.36,CART,0.1,LOCAL,6
14,-0.442487,5612.37,RANDOM,0.5,LOCAL,5
8,-0.442487,3238.88,CART,0.5,LOCAL,5
15,-0.442532,5968.96,CART,0.1,LOCAL,5
12,-0.442532,4977.28,RANDOM,0.1,LOCAL,5
10,-0.442789,4109.83,RANDOM,0.5,LOCAL,4
19,-0.442789,7490.74,CART,0.5,LOCAL,4


CPU times: user 18h 25min 34s, sys: 10min 57s, total: 18h 36min 32s
Wall time: 6h 32min 43s


In [10]:
ydf_models[0].describe()

trial,score,duration,categorical_algorithm,l2_regularization,growing_strategy,max_depth
1,-0.439309,899.175,CART,0,LOCAL,4
13,-0.439309,5991.1,RANDOM,0,LOCAL,4
16,-0.439498,7339.97,RANDOM,0,LOCAL,5
4,-0.439498,2194.13,CART,0,LOCAL,5
15,-0.439663,6844,CART,0.1,LOCAL,5
12,-0.439663,5516.88,RANDOM,0.1,LOCAL,5
6,-0.439689,3003.5,RANDOM,0.1,LOCAL,4
19,-0.439741,8439.64,CART,0.5,LOCAL,4
10,-0.439741,4598.18,RANDOM,0.5,LOCAL,4
7,-0.439809,3400.73,CART,0.5,LOCAL,6


In [11]:
ydf_models[1].describe()

trial,score,duration,categorical_algorithm,l2_regularization,growing_strategy,max_depth
15,-0.441671,6147.05,CART,0.1,LOCAL,5
12,-0.441671,4865.52,RANDOM,0.1,LOCAL,5
3,-0.441831,1268.46,RANDOM,0.5,LOCAL,6
7,-0.441831,2765.21,CART,0.5,LOCAL,6
10,-0.442025,3991.83,RANDOM,0.5,LOCAL,4
19,-0.442025,7478.09,CART,0.5,LOCAL,4
9,-0.442039,3626.85,CART,0.1,LOCAL,6
11,-0.442039,4394.51,RANDOM,0.1,LOCAL,6
4,-0.442043,1586.88,CART,0,LOCAL,5
16,-0.442043,6467.43,RANDOM,0,LOCAL,5


In [12]:
ydf_models[2].describe()

trial,score,duration,categorical_algorithm,l2_regularization,growing_strategy,max_depth
16,-0.442205,6376.36,RANDOM,0,LOCAL,5
4,-0.442205,1787.99,CART,0,LOCAL,5
11,-0.442285,4613.66,RANDOM,0.1,LOCAL,6
9,-0.442285,3732.36,CART,0.1,LOCAL,6
14,-0.442487,5612.37,RANDOM,0.5,LOCAL,5
8,-0.442487,3238.88,CART,0.5,LOCAL,5
15,-0.442532,5968.96,CART,0.1,LOCAL,5
12,-0.442532,4977.28,RANDOM,0.1,LOCAL,5
10,-0.442789,4109.83,RANDOM,0.5,LOCAL,4
19,-0.442789,7490.74,CART,0.5,LOCAL,4


# 3. 10 FOLD CV WITH SKLEARN TARGET ENCODING

In [13]:
# %%time

# MODEL_NAME = 'YDFlocal'
# FOLDS = 3
# print(f"\n🚀 Training {MODEL_NAME} with {FOLDS} Folds...")

# X = train[FEATURES]
# y = train[TARGET].astype(int)
# X_test = test[FEATURES]

# skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
# oof_preds  = np.zeros(len(train))
# test_preds = np.zeros(len(test))

# # feature_importances = pd.DataFrame()

# for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
#     X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
#     X_valid, y_valid = X.iloc[valid_idx].copy(), y.iloc[valid_idx]
#     X_test_fold = X_test.copy()

#     # Triple Sklearn Target Encoders (Auto, Strict 10, and Massive 100)
#     te_auto = TargetEncoder(shuffle=True, cv=5, smooth='auto', random_state=42)
#     te_10   = TargetEncoder(shuffle=True, cv=5, smooth=10.0, random_state=42)
#     te_100  = TargetEncoder(shuffle=True, cv=5, smooth=100.0, random_state=42)

#     X_train_enc_auto = te_auto.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
#     X_valid_enc_auto = te_auto.transform(X_valid[TARGET_ENCODE_COLS])
#     X_test_enc_auto  = te_auto.transform(X_test_fold[TARGET_ENCODE_COLS])

#     X_train_enc_10 = te_10.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
#     X_valid_enc_10 = te_10.transform(X_valid[TARGET_ENCODE_COLS])
#     X_test_enc_10  = te_10.transform(X_test_fold[TARGET_ENCODE_COLS])

#     X_train_enc_100 = te_100.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
#     X_valid_enc_100 = te_100.transform(X_valid[TARGET_ENCODE_COLS])
#     X_test_enc_100  = te_100.transform(X_test_fold[TARGET_ENCODE_COLS])

#     for i, col in enumerate(TARGET_ENCODE_COLS):
#         # Auto smoothing TE
#         X_train[f"{col}_TE_auto"] = X_train_enc_auto[:, i].astype('float32')
#         X_valid[f"{col}_TE_auto"] = X_valid_enc_auto[:, i].astype('float32')
#         X_test_fold[f"{col}_TE_auto"] = X_test_enc_auto[:, i].astype('float32')
        
#         # Strict (10.0) smoothing TE
#         X_train[f"{col}_TE_10"] = X_train_enc_10[:, i].astype('float32')
#         X_valid[f"{col}_TE_10"] = X_valid_enc_10[:, i].astype('float32')
#         X_test_fold[f"{col}_TE_10"] = X_test_enc_10[:, i].astype('float32')

#         # Massive (100.0) smoothing TE (Markus's method)
#         X_train[f"{col}_TE_100"] = X_train_enc_100[:, i].astype('float32')
#         X_valid[f"{col}_TE_100"] = X_valid_enc_100[:, i].astype('float32')
#         X_test_fold[f"{col}_TE_100"] = X_test_enc_100[:, i].astype('float32')
        
#         # Drop the original string column
#         X_train.drop(columns=[col], inplace=True)
#         X_valid.drop(columns=[col], inplace=True)
#         X_test_fold.drop(columns=[col], inplace=True)
        
#     # APPLY DYNAMIC PRUNING
#     if len(features_to_drop) > 0:
#         # Check X_train.columns because it contains both base features and the new _TE_ features
#         safe_to_drop = [c for c in features_to_drop if c in X_train.columns]
        
#         if len(safe_to_drop) > 0:
#             if fold == 1:
#                 print(f"   -> Successfully pruned {len(safe_to_drop)} features.")
                
#             X_train.drop(columns=safe_to_drop, inplace=True, errors='ignore')
#             X_valid.drop(columns=safe_to_drop, inplace=True, errors='ignore')
#             X_test_fold.drop(columns=safe_to_drop, inplace=True, errors='ignore')

#     if fold == 1:
#         display(X_train.head())

#     # -- TUNED YDF MODEL --
#     local_params = {
#         'task': ydf.Task.CLASSIFICATION,
#         'growing_strategy': "LOCAL",
#         'num_trees': 1000, # def=300
#         'max_depth': 4,
#         # 'shrinkage': 0.05, # def=0.1
#         # 'categorical_algorithm': "CART", # "CART", "RANDOM", 'ONE_HOT'
#         # 'sampling_method': 'RANDOM', # 'RANDOM', 'GOSS', 'SELGB'
#         # 'split_axis': , # 'AXIS_ALIGNED', 'SPARSE_OBLIQUE', 'MHLD_OBLIQUE'
#         # 'subsample': 0.95, # default RANDOM sampling
#         # 'goss_alpha': 0.2, # activates GOSS sampling
#         # 'l2_regularization': 1.0,
#         # 'early_stopping': 'NONE', # 'NONE', 'MIN_LOSS_FINAL', 'LOSS_INCREASE'
#         # 'early_stopping_initial_iteration': 10,
#         'early_stopping_num_trees_look_ahead': 50,
#         'random_seed': 42,
#     }
#     global_params = {
#         'task': ydf.Task.CLASSIFICATION,
#         'growing_strategy': "BEST_FIRST_GLOBAL",
#         'num_trees': 1000,
#         # 'max_num_nodes': None,
#         'shrinkage': 0.05,
#         # 'categorical_algorithm': 'RANDOM', # 'CART', 'RANDOM', 'ONE_HOT'
#         # 'sampling_method': 'RANDOM', # 'RANDOM', 'GOSS', 'SELGB'
#         # 'split_axis': , # 'AXIS_ALIGNED', 'SPARSE_OBLIQUE', 'MHLD_OBLIQUE'
#         # 'subsample': 0.95, # default RANDOM sampling
#         # 'goss_alpha': 0.2, # activates GOSS sampling
#         # 'l2_regularization': 1.0,
#         # 'early_stopping': 'MIN_LOSS_FINAL', # NONE, MIN_LOSS_FINAL, LOSS_INCREASE
#         # 'early_stopping_initial_iteration': 10,
#         'early_stopping_num_trees_look_ahead': 50,
#         'random_seed': 42,
#     }
#     clf = ydf.GradientBoostedTreesLearner(
#             **local_params if 'local' in MODEL_NAME else global_params,
#             label=TARGET,
#             # class_weights=get_class_weights(y_train) if use_weights else None,
#             # features=flat_features,   # 1. Semantic for categories goes here
#             # include_all_columns=True, # 2. Include all features for training
#         ).train(
#             ds=pd.concat([X_train, y_train], axis=1),
#             valid=pd.concat([X_valid, y_valid], axis=1),
#             verbose=1,
#         )

#     oof_preds[valid_idx] = clf.predict(X_valid)
#     test_preds += clf.predict(X_test_fold) / FOLDS

#     fold_auc = roc_auc_score(y_valid, oof_preds[valid_idx])
#     print(f"   --> Fold {fold} AUC: {fold_auc:.6f}/n")

# oof_auc = roc_auc_score(y, oof_preds)

# print("\n" + "="*45)
# print(f"🏆 {MODEL_NAME} OOF AUC: {oof_auc:.6f}")
# print("="*45)

In [14]:
# =============================================
# 🏆 YDFlocal OOF AUC: 0.945583
# =============================================
# CPU times: user 1h 18min 51s, sys: 51.8 s, total: 1h 19min 43s
# Wall time: 24min 36s

# =============================================
# 🏆 YDFglobal OOF AUC: 0.945431
# 🏆 YDFglobal OOF AUC: 0.945458 -> fixed digit extraction
# =============================================
# CPU times: user 42min 31s, sys: 45.6 s, total: 43min 16s
# Wall time: 13min 43s

In [15]:
# clf.analyze(pd.concat([X_valid, y_valid], axis=1))

# 4. SAVE SUBMISSION AND OOF PREDICTIONS

In [16]:
# print("\n" + "="*45)
# print(f"🏆 {MODEL_NAME} FINAL OOF AUC: {oof_auc:.6f}\n")
# print("="*45)

# MODEL_NAME = f"{MODEL_NAME}_Triple_TE"

# # Save Kaggle Submission (Averaged Test Predictions)
# submission[TARGET] = test_preds
# submission.to_csv(f"submission_{MODEL_NAME}_{oof_auc:.6f}.csv", index=False)
# print(f"💾 Saved 'submission_{MODEL_NAME}_{oof_auc:.6f}.csv'")

# # Save OOF Predictions (for ensembling meta-model)
# # oof_df = pd.DataFrame({'id': train['id'], 'OOF_Pred': oof_preds})
# # oof_df.to_csv(f'oof_{MODEL_NAME}.csv', index=False)
# np.save(f"oof_{MODEL_NAME}_{oof_auc:.6f}.npy", oof_preds)
# print(f"💾 Saved 'oof_{MODEL_NAME}_{oof_auc:.6f}.npy'")

# # Save Raw Test Predictions (for ensembling)
# # test_df = pd.DataFrame({'id': test['id'], TARGET: test_preds})
# # test_df.to_csv(f'test_{MODEL_NAME}.csv', index=False)
# np.save(f"test_{MODEL_NAME}_{oof_auc:.6f}.npy", test_preds)
# print(f"💾 Saved 'test_{MODEL_NAME}_{oof_auc:.6f}.npy'")

# # feature_importances = feature_importances.sort_values(by='importance', ascending=False).reset_index(drop=True)
# # # feature_importances.to_csv(f'feature_importance_{MODEL_NAME}.csv', index=False)
# # print(f"💾 Saved 'feature_importance_{MODEL_NAME}.csv'")

# # # Print the top 10 and bottom 10 features
# # print("\n🔥 Top 10 Features:")
# # print(feature_importances.head(10))
# # print("\n🧊 Bottom 10 Features (Candidates for Pruning):")
# # print(feature_importances.tail(10))

In [17]:
# buyers = test_preds[test_preds >= 0.5]
# non_buyers = test_preds[test_preds < 0.5]

# sns.histplot(buyers, label="Buyers")
# sns.histplot(non_buyers, label="Non-Buyers")
# plt.legend()
# plt.show()

In [18]:
# !rm -r /kaggle/working/